# PDF Parsing với LangChain

1. Read text  from PDF directly
2. Use LLM to read PDF directly
3. Convert PDF to image and use LLM to read text
4. Parse CV into structured data

## Setup - Import Libraries

In [1]:
import os
from pathlib import Path
from typing import Optional, List, Dict, Any
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_community.document_loaders import PyPDFLoader
from config import settings

# For PDF to image conversion
try:
    from pdf2image import convert_from_path
    from PIL import Image
    PDF2IMAGE_AVAILABLE = True
except ImportError:
    PDF2IMAGE_AVAILABLE = False
    print("Warning: pdf2image not installed. Install with: uv add pdf2image pillow")

# Initialize LLM
llm = ChatOpenAI(
    model=settings.LLM_CHAT_MODEL,
    api_key=settings.LLM_API_KEY,
    base_url=settings.LLM_BASE_URL,
    temperature=0
)

# Path to CV PDF
CV_PATH = "data/cv.pdf"
print(f"CV file path: {CV_PATH}")

CV file path: data/cv.pdf


## Function 1: Đọc Text từ PDF (Cơ bản)

Sử dụng PyPDFLoader từ LangChain để đọc text từ PDF.

In [2]:
def read_pdf_text(pdf_path: str) -> str:
   
    try:
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        text = "\n\n".join([doc.page_content for doc in documents])
        
        
        return text
    except Exception as e:
        print(f"Error: {e}")
        return ""

# Test function
pdf_text = read_pdf_text(CV_PATH)
print(f"\nPreview (first 500 chars):\n{pdf_text[:500]}...")


Preview (first 500 chars):
Hoàng Đình Hùng
Khoa học máy tính
Trường Công nghệ Thông tin và Truyền thông
Đại học Bách Khoa Hà Nội
/ne+84-375838482
/enve♀ehung.hd210399@sis.hust.edu.vn
/gtbGitHub Profile
/♀nednLinkedin Profile
HỌC V ẤN
Đại học Bách Khoa Hà Nội 2021 - hiện tại
Khoa học máy tính CPA: 3.92/4
THÀNH TÍCH V À CHỨNG CHỈ
Học bổng khuyến khích học tập học kỳ 2, 4,5,6, 7, 8. 6/2022-12/2023
Học bổng MB năm học 2023-2024 12/2023
Học bổng MB năm học 2024-2025 12/2024
Giải khuyến khích cuộc thi viết "Màu cờ dưới ánh bình...


## Function 2: Dùng LLM đọc PDF trực tiếp

Đọc text từ PDF rồi dùng LLM để phân tích và trích xuất thông tin có cấu trúc.

In [3]:
class CVInfo(BaseModel):
    """Cấu trúc thông tin CV"""
    name: str = Field(description="Tên ứng viên")
    email: Optional[str] = Field(description="Email", default=None)
    phone: Optional[str] = Field(description="Số điện thoại", default=None)
    skills: List[str] = Field(description="Danh sách kỹ năng")
    experience: List[Dict[str, str]] = Field(description="Kinh nghiệm làm việc")
    education: List[Dict[str, str]] = Field(description="Học vấn")
    summary: Optional[str] = Field(description="Tóm tắt", default=None)

def parse_cv_with_llm(pdf_path: str) -> Dict[str, Any]:
    pdf_text = read_pdf_text(pdf_path)
    
    if not pdf_text:
        return {}
    
    parser = PydanticOutputParser(pydantic_object=CVInfo)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Bạn là chuyên gia phân tích CV. Hãy trích xuất thông tin từ CV dưới đây."),
        ("human", "CV Content:\n{content}\n\n{format_instructions}")
    ]).partial(format_instructions=parser.get_format_instructions())
    
    chain = prompt | llm | parser
    try:
        result = chain.invoke({"content": pdf_text[:4000]})  # Limit to avoid token limit
        return result.model_dump()
    except Exception as e:
        print(f"Error: {e}")
        return {}


cv_data = parse_cv_with_llm(CV_PATH)
import json
print(json.dumps(cv_data, indent=2, ensure_ascii=False))

{
  "name": "Hoàng Đình Hùng",
  "email": "hung.hd210399@sis.hust.edu.vn",
  "phone": "+84-375838482",
  "skills": [
    "AI Engineer",
    "Back End Development",
    "Project Management",
    "Natural Language Processing (NLP)",
    "Big Data Analysis",
    "LaTeX",
    "English (TOEIC 815)"
  ],
  "experience": [
    {
      "organization": "Foundation Modals Labs - Trung tâm nghiên cứu quốc tế BKAI",
      "role": "Research Assistant",
      "duration": "05/2024 - hiện tại",
      "description": "Tham gia dự án Hệ thống tư vấn pháp luật."
    },
    {
      "organization": "Trường Công nghệ Thông tin và Truyền thông - Đại học Bách Khoa Hà Nội",
      "role": "Trưởng mảng Đại số, CLB Hỗ trợ học tập",
      "duration": "04/2022 - 10/2024",
      "description": "Trợ giảng lớp đại cương môn Đại số và Giải tích 2; Tổ chức khóa dạy học cấp chứng chỉ Latex."
    },
    {
      "project": "Hệ thống hỏi chatbot của Viettel Software",
      "duration": "10/2024 - 02/2025",
      "description

In [4]:
import fitz  # PyMuPDF
from PIL import Image
from typing import List

def pdf_to_images(pdf_path: str, dpi: int = 200) -> List[Image.Image]:
    images = []
    zoom = dpi / 72 
    mat = fitz.Matrix(zoom, zoom)

    doc = fitz.open(pdf_path)
    for page in doc:
        pix = page.get_pixmap(matrix=mat)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        images.append(img)

    print(f"Converted {len(images)} pages to images")
    return images


In [5]:
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.messages import HumanMessage
import base64
from io import BytesIO

def read_image_with_llm(image: Image.Image, page_num: int) -> str:
   
    llm = ChatOpenAI(
        model=settings.LLM_CHAT_MODEL,
        api_key=settings.LLM_API_KEY,
        base_url=settings.LLM_BASE_URL,
        temperature=0,
    )
    
    # Convert image to base64
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    img_base64 = base64.b64encode(buffered.getvalue()).decode()
    
    # Create message
    message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": "Hãy đọc tất cả text trong ảnh này. Trả về text thuần túy, không cần format đặc biệt."
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{img_base64}"
                }
            }
        ]
    )
    
    try:
        response = llm.invoke([message])
        return response.content
    except Exception as e:
        print(f"Error: {e}")
        return ""

def parse_pdf_images_and_aggregate(pdf_path: str) -> Dict[str, Any]:
   
    if not PDF2IMAGE_AVAILABLE:
        print("PDF2IMAGE not installed  ")
        return {}
    
    
    images = pdf_to_images(pdf_path)
    print(f"Convert {len(images)} pages to images")
    if not images:
        return {}
  
    all_texts = []
    for i, image in enumerate(images, 1):
        print(f"  Processing page {i}/{len(images)}...")
        text = read_image_with_llm(image, i)
        all_texts.append(f"=== Page {i} ===\n{text}")
    
    combined_text = "\n\n".join(all_texts)
    
        
    
    parser = PydanticOutputParser(pydantic_object=CVInfo)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Bạn là chuyên gia phân tích CV. Hãy trích xuất thông tin từ text CV dưới đây."),
        ("human", "CV Text (đã đọc từ ảnh):\n{content}\n\n{format_instructions}")
    ]).partial(format_instructions=parser.get_format_instructions())
    
    chain = prompt | llm | parser
    
    try:
        result = chain.invoke({"content": combined_text[:8000]})  # Limit tokens
        return result.model_dump()
    except Exception as e:
        print(f"Error: {e}")
        return {"raw_text": combined_text[:2000]}

# Test function
if PDF2IMAGE_AVAILABLE:
    cv_data_aggregated = parse_pdf_images_and_aggregate(CV_PATH)
    import json
    print(json.dumps(cv_data_aggregated, indent=2, ensure_ascii=False))
else:
    print("PDF2IMAGE not installed  ")

Converted 2 pages to images
Convert 2 pages to images
  Processing page 1/2...
  Processing page 2/2...
{
  "name": "Hoàng Đình Hùng",
  "email": "hung.hd210399@sis.hust.edu.vn",
  "phone": "+84-375838482",
  "skills": [
    "Khoa học máy tính",
    "AI Engineer",
    "Back End Development",
    "Project Management",
    "NLP (Natural Language Processing)",
    "AI Agent",
    "Xử lý và phân tích dữ liệu lớn",
    "Latex",
    "Tiếng Anh (TOEIC 815/990)",
    "Giảng dạy/Trợ giảng (Đại số, Giải tích)"
  ],
  "experience": [
    {
      "organization": "CLB Hỗ trợ học tập - Trường CNTT&TT - ĐHBKHN",
      "role": "Trưởng mảng Đại số / Thành viên tích cực",
      "duration": "04/2022 - 10/2024",
      "description": "Trợ giảng lớp đại cương môn Đại số và Giải tích 2; Tham gia tổ chức khóa dạy học cấp chứng chỉ Latex."
    },
    {
      "organization": "Foundation Modals Labs - Trung tâm nghiên cứu quốc tế BKAI",
      "role": "Research Assistant",
      "duration": "05/2024 - hiện tại",
